In [1]:
import sys
import os

# This adds the parent directory (..) to the Python path
sys.path.append(os.path.abspath('..'))

from datasets import load_dataset
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from typing import List

from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    AutoConfig,
    set_seed,
)

from models import CloseTrack_Predictor

from utils import ( 
    compute_metrics, 
    load_model_params,
    load_data_paths,
    preprocess_dataset,
    cleanup_trainer_memory
)

In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
DATA_DIR = Path("../data/")
MODELS_DIR = Path("../models/")
sm_addAttn_model_path = MODELS_DIR / 'mlp_ScalarMix_AddAttn_xx'
sm_selfAttn_model_path = MODELS_DIR / 'mlp_ScalarMix_SelfAttn_xx'
selfAttn_model_path = MODELS_DIR / "mlp_SelfAttn_xx"
addAttn_model_path = MODELS_DIR / 'mlp_AddAttn_xx'
model_params_path = MODELS_DIR / "model_parameters.csv"

In [8]:
model = CloseTrack_Predictor.from_pretrained(
    "HenryLi0925/ScalarMix_AddAttn_mlp",
    layer_wise = 'ScalarMix',
    token_wise = 'AddAttn'
)

config.json:   0%|          | 0.00/739 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

In [6]:
sm_addAttn_mlp = CloseTrack_Predictor.from_pretrained(
                    sm_addAttn_model_path,
                    layer_wise = 'ScalarMix',
                    token_wise = 'AddAttn'
                    )

In [7]:
sm_addAttn_mlp.push_to_hub("HenryLi0925/ScalarMix_AddAttn_mlp")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/HenryLi0925/ScalarMix_AddAttn_mlp/commit/3dfa3010202eb17ca6736dc34e396d7c096e190e', commit_message='Upload CloseTrack_Predictor', commit_description='', oid='3dfa3010202eb17ca6736dc34e396d7c096e190e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HenryLi0925/ScalarMix_AddAttn_mlp', endpoint='https://huggingface.co', repo_type='model', repo_id='HenryLi0925/ScalarMix_AddAttn_mlp'), pr_revision=None, pr_num=None)

In [4]:
for name, params in sm_addAttn_mlp.named_parameters():
    if "ScalarMix" in name:
        print(f"{name}: {params}")

ScalarMix.temp: Parameter containing:
tensor([3.6861e-29], requires_grad=True)
ScalarMix.scalar_weights: Parameter containing:
tensor([1.7994e-29, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.3303e+22, 3.9895e-11,
        7.1855e+22, 7.1955e+28, 7.1481e+22, 2.0997e+26, 3.5243e+33, 4.3534e+24,
        1.8006e-29], requires_grad=True)


In [26]:
F.softmax(sm_addAttn_mlp.ScalarMix.scalar_weights, dim=0)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       grad_fn=<SoftmaxBackward0>)

In [13]:
sm_selfAttn_mlp = CloseTrack_Predictor.from_pretrained(
                    sm_selfAttn_model_path,
                    layer_wise = 'ScalarMix',
                    token_wise = 'SelfAttn'
                    )

In [15]:
sm_selfAttn_mlp.push_to_hub("HenryLi0925/ScalarMix_SelfAttn_mlp")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/HenryLi0925/ScalarMix_SelfAttn_mlp/commit/4e95ee27ca6cdc157191966d015eda0b7acf0bd3', commit_message='Upload CloseTrack_Predictor', commit_description='', oid='4e95ee27ca6cdc157191966d015eda0b7acf0bd3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HenryLi0925/ScalarMix_SelfAttn_mlp', endpoint='https://huggingface.co', repo_type='model', repo_id='HenryLi0925/ScalarMix_SelfAttn_mlp'), pr_revision=None, pr_num=None)

In [14]:
for name, params in sm_selfAttn_mlp.named_parameters():
    if "ScalarMix" in name:
        print(f"{name}: {params}")

ScalarMix.temp: Parameter containing:
tensor([5.6566e-29], requires_grad=True)
ScalarMix.scalar_weights: Parameter containing:
tensor([1.1040e-29, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.3303e+22, 4.0122e-11,
        7.1855e+22, 7.1955e+28, 3.0196e+32, 7.7465e+31, 7.2814e+34, 6.3631e+28,
        8.0111e+20], requires_grad=True)


In [63]:
F.softmax(sm_selfAttn_mlp.ScalarMix.scalar_weights)

/tmp/ipykernel_3804/3478425429.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  F.softmax(sm_selfAttn_mlp.ScalarMix.scalar_weights)


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
baseline_model_path = MODELS_DIR / 'baseline_open_xx'
baseline_model = AutoModelForSequenceClassification.from_pretrained(
                        baseline_model_path
                    )

In [33]:
data_files = load_data_paths(DATA_DIR, 'xx', "finetune")
hf_dataset = load_dataset("csv", data_files=data_files)

In [34]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base', use_fast=True)
cols_to_merge = "L1_source_word; L1_context; en_target_clue; en_target_word".split("; ")
sep_token = f" {tokenizer.sep_token} " if tokenizer.sep_token else " "

In [35]:
preprocessed_ds = preprocess_dataset(hf_dataset, cols_to_merge, sep_token)

# Tokenize dataset
tokenized_ds = preprocessed_ds.map(
    lambda x: tokenizer(x["input_text"], truncation=True),
    batched=True,
    desc="Tokenizing input text"
)

In [36]:
# Itemize L1 
l1_to_idx = {l1:i for i, l1 in enumerate(["es", "de", "cn"])}
tokenized_ds = tokenized_ds.map(
    lambda x: {"l1_encode": [l1_to_idx[val] for val in x['L1']]},
    batched=True,
    remove_columns=['L1'],
    desc="Itemizing L1"
)

In [28]:
# Extract L1 embeddings
l1_to_iso = {"es": "spa",
"de": 'deu',
"cn": "zho"}
l1_to_embed = {l1: learned_database[l1_to_iso[l1]].tolist() for l1 in ["es", "de", "cn"]}
tokenized_ds = tokenized_ds.map(
    lambda x: {"l1_embed": [l1_to_embed[val] for val in x['L1']]},
    batched=True,
    remove_columns=['L1'],
    desc="L1 Embeddings"
)

In [37]:
train_ds = tokenized_ds['train']

In [38]:
# Initialize the collator with your tokenizer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_dataloader = DataLoader(train_ds.remove_columns(['input_text']), batch_size=64, collate_fn = data_collator)

In [39]:
data = next(iter(train_dataloader))

In [115]:
baseline_output = baseline_model(input_ids = data['input_ids'], attention_mask = data['attention_mask'], output_hidden_states=True, output_attentions=True)
baseline_logits, baseline_hidden, baseline_attention = baseline_output['logits'], baseline_output['hidden_states'], baseline_output['attentions']

XLMRobertaSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


In [42]:
output = sm_addAttn_mlp(*(data['input_ids'], data['attention_mask'], data['labels']), output_attentions=True)
attention_masks = data['attention_mask']
logits, hiddens, attentions, attn_weights = output['logits'], output['hidden_states'], output['attentions'], output['token_attn_weights']
hiddens = torch.stack(hiddens,0)

In [52]:
tokenizer.decode(data['input_ids'][0])

'<s> lapso</s> El eclipse solar fue visible durante un breve lapso de tiempo.</s> s___</s> span</s><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [49]:
attn_weights[0,:]

tensor([0.0098, 0.0727, 0.0661, 0.0103, 0.0397, 0.0459, 0.0463, 0.0479, 0.0488,
        0.0457, 0.0540, 0.0507, 0.0334, 0.0370, 0.0561, 0.0430, 0.0171, 0.0456,
        0.0434, 0.0104, 0.0452, 0.0527, 0.0373, 0.0304, 0.0104, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SliceBackward0>)

In [54]:
seq_len = hiddens[0].size(-2)
input_dim = hiddens[0].size(-1)
normed_hiddens = F.layer_norm(hiddens, normalized_shape=(seq_len, input_dim))

In [71]:
torch.mean(torch.stack(attentions,0), dim=(-3,-2,-1))

tensor([[0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294],
        [0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294, 0.0294,
         0

In [52]:
torch.mean(hiddens, dim=(-1))

tensor([[[-0.0118, -0.0009, -0.0023,  ...,  0.0221,  0.0221,  0.0221],
         [-0.0118, -0.0065, -0.0047,  ...,  0.0221,  0.0221,  0.0221],
         [-0.0118, -0.0026, -0.0066,  ...,  0.0221,  0.0221,  0.0221],
         ...,
         [-0.0118, -0.0037, -0.0023,  ...,  0.0221,  0.0221,  0.0221],
         [-0.0118, -0.0036, -0.0024,  ...,  0.0221,  0.0221,  0.0221],
         [-0.0118, -0.0007, -0.0066,  ...,  0.0221,  0.0221,  0.0221]],

        [[-0.0119, -0.0269, -0.0371,  ..., -0.0174, -0.0174, -0.0174],
         [-0.0118, -0.0231, -0.0254,  ..., -0.0173, -0.0173, -0.0173],
         [-0.0118, -0.0237, -0.0201,  ..., -0.0193, -0.0193, -0.0193],
         ...,
         [-0.0119, -0.0332, -0.0355,  ..., -0.0185, -0.0185, -0.0185],
         [-0.0119, -0.0294, -0.0417,  ..., -0.0193, -0.0193, -0.0193],
         [-0.0118, -0.0261, -0.0197,  ..., -0.0172, -0.0172, -0.0172]],

        [[ 0.0259,  0.0132,  0.0053,  ..., -0.0013, -0.0013, -0.0013],
         [ 0.0259,  0.0066,  0.0106,  ..., -0

In [44]:
torch.var(hiddens, dim=(1,2,3))

tensor([0.0825, 0.2139, 0.5298, 0.6543, 0.5787, 0.5664, 0.5423, 0.5475, 0.6025,
        0.6291, 0.8104, 0.8494, 0.4727], grad_fn=<VarBackward0>)

In [58]:
torch.mean(normed_hiddens, dim=(-2,-1))

tensor([[ 1.0518e-08, -7.5967e-09,  1.1103e-08, -4.0905e-09, -4.0905e-09,
         -5.8436e-10,  1.2272e-08,  4.2366e-09,  2.3374e-09,  5.8436e-09,
          2.3374e-09,  1.7531e-09,  1.0226e-08, -8.1810e-09,  7.0123e-09,
         -4.0905e-09, -9.7880e-09, -3.2140e-09, -5.8436e-10, -7.0123e-09,
          6.7201e-09,  9.0576e-09, -8.1810e-09,  6.4280e-09,  7.0123e-09,
          1.2856e-08,  2.3374e-09, -7.3045e-09, -1.4171e-08,  5.8436e-10,
          5.8436e-09,  7.5967e-09, -2.0453e-09, -1.1395e-08, -1.0811e-08,
          2.9218e-09,  6.4280e-09,  1.6070e-08,  5.8436e-10, -2.3374e-09,
         -1.1103e-08, -3.5062e-09,  6.4280e-09,  7.3045e-09, -8.7654e-10,
          1.4609e-08, -5.5514e-09, -4.6749e-09,  9.3497e-09, -1.0518e-08,
          7.5967e-09, -2.9218e-10, -1.4609e-08, -8.7654e-09, -7.0123e-09,
         -1.6362e-08,  3.2140e-09, -1.1395e-08,  7.5967e-09,  9.9341e-09,
          1.1103e-08,  1.0518e-08, -4.6749e-09, -2.9218e-09],
        [-9.6419e-09,  6.4280e-09,  4.6749e-09,  5

In [53]:
torch.var(normed_hiddens, dim=(-1))

tensor([[[1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012],
         [1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012],
         [1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012],
         ...,
         [1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012],
         [1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012],
         [1.0011, 1.0011, 1.0011,  ..., 1.0012, 1.0012, 1.0012]],

        [[1.0013, 1.0013, 1.0013,  ..., 1.0012, 1.0012, 1.0012],
         [1.0013, 1.0012, 1.0013,  ..., 1.0012, 1.0012, 1.0012],
         [1.0013, 1.0013, 1.0012,  ..., 1.0012, 1.0012, 1.0012],
         ...,
         [1.0013, 1.0013, 1.0013,  ..., 1.0012, 1.0012, 1.0012],
         [1.0013, 1.0013, 1.0013,  ..., 1.0012, 1.0012, 1.0012],
         [1.0013, 1.0013, 1.0012,  ..., 1.0012, 1.0012, 1.0012]],

        [[1.0013, 1.0013, 1.0013,  ..., 1.0013, 1.0013, 1.0013],
         [1.0013, 1.0013, 1.0013,  ..., 1.0013, 1.0013, 1.0013],
         [1.0013, 1.0013, 1.0013,  ..., 1.0013, 1.0013, 1.

In [17]:
torch.stack(attention, 0).shape

torch.Size([12, 64, 12, 34, 34])

In [58]:
output = sm_selfAttn_mlp(*(data['input_ids'], data['attention_mask'], data['labels']), output_attentions=True)
attention_masks = data['attention_mask']
logits, hiddens, attentions, attn_weights = output['logits'], output['hidden_states'], output['attentions'], output['token_attn_weights']
hiddens = torch.stack(hiddens,0)

In [62]:
attn_weights[0,:]

tensor([0.0060, 0.0827, 0.0506, 0.0576, 0.0253, 0.0329, 0.0101, 0.0251, 0.0201,
        0.0481, 0.0284, 0.0382, 0.0198, 0.0384, 0.0864, 0.0443, 0.0120, 0.0370,
        0.0387, 0.0494, 0.0491, 0.0276, 0.0599, 0.0573, 0.0548, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SliceBackward0>)

In [92]:
output_roberta = model.roberta(input_ids = data['input_ids'], attention_mask = data['attention_mask'], output_hidden_states = True)

In [120]:
baseline_roberta = baseline_model.roberta(input_ids = data['input_ids'], attention_mask = data['attention_mask'], output_hidden_states = True)

In [121]:
baseline_roberta['last_hidden_state'][0]

tensor([[-0.7172,  1.2049, -0.1322,  ..., -0.9779, -0.3234,  0.7841],
        [-1.3320,  1.4702, -0.0206,  ..., -0.2785, -0.1526,  0.5349],
        [-1.2018,  1.5072,  0.2048,  ..., -0.3371, -0.0695,  0.6125],
        ...,
        [-0.7278,  1.1619, -0.2060,  ..., -1.0484, -0.4093,  0.8232],
        [-0.7278,  1.1619, -0.2060,  ..., -1.0484, -0.4093,  0.8232],
        [-0.7278,  1.1619, -0.2060,  ..., -1.0484, -0.4093,  0.8232]],
       grad_fn=<SelectBackward0>)

In [106]:
output_roberta['last_hidden_state'][0]

tensor([[ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698],
        [ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698],
        [ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698],
        ...,
        [ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698],
        [ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698],
        [ 0.0327,  0.0820,  0.0317,  ..., -0.0961,  0.0059,  0.0698]],
       grad_fn=<SelectBackward0>)

In [44]:
raw_token = output.hidden_states[-1][:,0,:].detach().numpy()

In [45]:
mixed_token = model.ScalarMix(token).detach().numpy()

In [78]:
raw_token

array([[ 0.03273812,  0.08195494,  0.03174688, ..., -0.09611104,
         0.00585331,  0.06977898],
       [ 0.03273809,  0.08195499,  0.03174692, ..., -0.09611112,
         0.0058533 ,  0.06977906],
       [ 0.03273809,  0.08195499,  0.03174693, ..., -0.09611107,
         0.00585332,  0.06977908],
       ...,
       [ 0.03273808,  0.08195499,  0.03174686, ..., -0.09611107,
         0.0058533 ,  0.06977907],
       [ 0.03273808,  0.08195496,  0.03174694, ..., -0.096111  ,
         0.00585336,  0.06977899],
       [ 0.03273807,  0.08195497,  0.03174689, ..., -0.09611108,
         0.00585331,  0.06977903]], shape=(64, 768), dtype=float32)

In [77]:
mixed_token

array([[-1.2260093e-04,  9.4297656e-04, -1.6664302e-04, ...,
         3.1065763e-04, -1.2983332e-03,  6.9944879e-05],
       [-1.2260089e-04,  9.4297656e-04, -1.6664302e-04, ...,
         3.1065763e-04, -1.2983332e-03,  6.9944879e-05],
       [-1.2260089e-04,  9.4297656e-04, -1.6664302e-04, ...,
         3.1065760e-04, -1.2983332e-03,  6.9944879e-05],
       ...,
       [-1.2260083e-04,  9.4297656e-04, -1.6664302e-04, ...,
         3.1065763e-04, -1.2983332e-03,  6.9944916e-05],
       [-1.2260083e-04,  9.4297674e-04, -1.6664302e-04, ...,
         3.1065763e-04, -1.2983332e-03,  6.9944879e-05],
       [-1.2260089e-04,  9.4297656e-04, -1.6664302e-04, ...,
         3.1065763e-04, -1.2983332e-03,  6.9944879e-05]],
      shape=(64, 768), dtype=float32)

In [54]:
input_dim = tensors.shape[-1]
num_not_masks = torch.sum(attention_masks, 1) * input_dim
b_masks = attention_masks.expand((13, -1, -1)).unsqueeze(-1)

In [70]:
masked_pt = b_masks * tensors

In [83]:
mean = torch.sum(masked_pt, dim = -2) / num_not_masks[None,:,None]
variance = torch.sum(((masked_pt - mean.unsqueeze(-2)) * b_masks) ** 2, dim = -2) / num_not_masks[None,:,None]
output = (masked_pt - mean.unsqueeze(-2)) / torch.sqrt(variance.unsqueeze(-2) + 1e-13) # ()

In [87]:
variance.size()

torch.Size([13, 64, 768])

In [14]:
input['input_ids'].size()

torch.Size([1, 25])

In [15]:
hidden[0][:, 0, :].size()

torch.Size([1, 768])

In [18]:
[h[:, 0, :] for h in hidden]

[tensor([[-1.4780e-01,  1.4928e-01, -1.4276e-01, -1.2172e-02,  2.5293e-01,
          -1.9234e-01,  1.6293e-01, -1.3027e-01,  7.4191e-02, -1.6746e-01,
          -1.6725e-02, -1.5345e-01, -2.6416e-01,  2.6388e-01, -1.1311e-01,
           1.1369e-01,  2.1805e-01, -2.1000e-01,  1.0747e-01,  7.1098e-02,
          -2.1057e-01, -1.6179e-01, -4.4164e-01, -3.7972e-02, -1.9344e-01,
          -8.0523e-02, -2.2804e-01,  9.5334e-02, -2.6768e-02,  2.8499e-01,
           2.5719e-02,  1.1740e-01, -8.5311e-02, -8.9176e-02, -1.7004e-01,
          -2.3630e-01,  1.1558e-01,  2.4526e-01,  9.4816e-02,  8.1019e-02,
          -3.8397e-02, -1.8147e-01, -1.9058e-02,  2.2326e-01,  3.4707e-01,
           6.4248e-02,  4.6656e-02,  2.4152e-02, -2.8043e-01, -5.4594e-02,
          -1.4857e-01,  2.3715e-01, -8.1620e-03, -7.6012e-02,  1.2004e-01,
          -3.3915e-02, -1.2303e-01, -6.2855e-02, -9.5637e-02,  8.4924e-02,
          -5.6988e-01,  1.0345e-01,  8.0037e-02,  8.5892e-02, -6.2466e-02,
           2.3411e-02,  6

In [123]:
sum(seq).size()

torch.Size([1, 64, 1, 768])

In [144]:
class ScalarMixing(nn.Module):
    def __init__(
        self, 
        mixture_size: int=13, 
        do_layer_norm: bool = False,
        initial_scalar_parameters: List[float] = None,
        trainable: bool = True) -> None:
        super().__init__()

        self.mixture_size = mixture_size
        self.do_layer_norm = do_layer_norm

        if initial_scalar_parameters is None:
            initial_scalar_parameters = [0.0] * mixture_size
        
        self.softmax = nn.Softmax(0)

        self.gamma = nn.Parameter(
            torch.FloatTensor([1.0]), 
            requires_grad=trainable
        )

        self.scalar_weights = nn.ParameterList(
            [
                nn.Parameter(
                    torch.FloatTensor(
                        [initial_scalar_parameters[i]]
                        ), 
                        requires_grad=trainable
                    )
                for i in range(mixture_size)
            ]
        )
    
    def forward(
        self, 
        tensors: List[torch.Tensor],
        mask: torch.bool=None
        ) -> torch.Tensor:
        # gamma * sum(softmax(weight) * tensor, dim=1)

        normed_weights = self.softmax(
            torch.cat([w for w in self.scalar_weights]) # (mixture_size)
            ) # (mixture_size)
        
        def _do_layer_norm(tensor, broadcast_mask, num_elements_not_masked, eps = 1e-13):
            # (x - mean) / sqrt(variance + eps)
            masked_pt = tensor * broadcast_mask
            mean = torch.sum(masked_pt) / num_elements_not_masked
            variance = torch.sum(((masked_pt - mean) * broadcast_mask) **2) / num_elements_not_masked
            return (masked_pt - mean) / torch.sqrt(variance + eps)

        seq = []
        if self.do_layer_norm:
            assert mask is not None
            # mask determines how many elements in sequence per batch we want to be valid 
            broadcast_mask = mask.unsqueeze(-1)
            # input_dim = hidden_size for each sequence.
            input_dim = tensors[0].size(-1)
            # number of elements not masked = total number of valid elements * hidden_size (e.g. 768) 
            number_elements_not_masked = torch.sum(mask) * input_dim
            for weights, tensor in zip(normed_weights, tensors):
                seq.append(weights * _do_layer_norm(tensor, broadcast_mask, number_elements_not_masked))

        else:
            for weights, tensor in zip(normed_weights, tensors):
                seq.append(weights * tensor)
        
        # output size: (1, batch_size, seq_len, hidden_size)
        return self.gamma * sum(seq)


In [ ]:
from transformers import AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput

class Ensemble_MLP(PreTrainedModel):
    config_class = AutoConfig
    def __init__(
        self, 
        config, 
        latent_dim = 64, 
        dropout: float = 0.1,
        full_sequence: bool = False
        ):

        super().__init__(config)
        
        self.config = config
        self.latent_dim = latent_dim
        self.full_sequence = full_sequence

        # add_pooling_layer=False to get raw hidden states
        self.backbone = AutoModel.from_config(config, add_pooling_layer=False)

        # full_sequence = False, use only <CLS> or <s> token for classification
        # no need for layer norm if not full_sequence
        do_layer_norm = full_sequence

        self.ScalarMix = ScalarMixing(
            config.num_hidden_layers + 1,
            do_layer_norm = do_layer_norm
            )

        self.mlp = nn.Sequential(
            nn.Linear(config.hidden_size, latent_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(latent_dim, config.num_labels)
        )

        # Initialize weights (Hugging Face standard)
        self.post_init()

    def forward(
        self,
        input_ids,
        attention_masks,
        labels,
        **kwargs) -> SequenceClassifierOutput:

        outputs = self.backbone(input_ids, attention_masks, output_hidden_states=True, **kwargs)

        if not self.full_sequence:
            token = [layer[:, 0, :] for layer in outputs.hidden_states]
            mixed_token = self.ScalarMix(token) # (batch, hidden)
        else:
            seq_tokens = [layer for layer in seq_output.hidden_states]
            mixed_seq = self.ScalarMix(seq_tokens, mask = attention_mask) # (batch, seq_len, hidden)
            
            # perform mean pooling if using full sequence 
            boardcast_mask = attention_mask.unsqueeze(-1).float() # (batch, seq_len, 1); float type for numerical stability
            sum_embed = torch.sum(mixed_seq * boardcast_mask, dim=1)
            sum_mask = torch.sum(boardcast_mask, dim=1).clamp(min=1e-9) # clamp to avoid zero-division due to all padding
            mixed_token = sum_embed/sum_mask # --> (batch, hidden)


        logits = self.mlp(mixed_token)   

        loss = None
        if labels is not None:     
            loss_fct = MSELoss(reduction='mean')
            loss = loss_fct(logits.squeeze(), labels.squeeze())
            
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=seq_output.hidden_states,
            attentions=seq_output.attentions,
        )

In [10]:
model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=1)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [119]:
model.config

XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}

In [120]:
outputs = model.roberta(input_ids = data['input_ids'], attention_mask = data['attention_mask'], output_attentions = True)

In [121]:
outputs.last_hidden_state

tensor([[[ 6.9579e-02,  1.8491e-01,  4.6383e-02,  ..., -1.5282e-01,
           1.2956e-01, -1.7178e-02],
         [-1.0588e-01,  1.0540e-02, -4.6108e-02,  ..., -1.8441e-01,
           3.2443e-02, -9.3319e-02],
         [-3.5748e-02, -1.2386e-01, -3.6218e-02,  ..., -1.0268e-01,
           9.0011e-02, -3.9028e-02],
         ...,
         [ 5.5112e-02,  2.0745e-01, -1.1017e-01,  ..., -4.2840e-01,
          -3.8940e-02,  8.4393e-02],
         [ 5.5112e-02,  2.0745e-01, -1.1017e-01,  ..., -4.2840e-01,
          -3.8940e-02,  8.4393e-02],
         [ 5.5112e-02,  2.0745e-01, -1.1017e-01,  ..., -4.2840e-01,
          -3.8940e-02,  8.4393e-02]],

        [[ 1.0371e-01,  1.6501e-01,  7.6258e-02,  ..., -1.3671e-01,
           1.2152e-01, -6.6367e-02],
         [-3.5850e-02,  3.4592e-02,  2.1359e-02,  ...,  1.7256e-01,
           1.1027e-01, -2.9847e-02],
         [-3.6444e-02, -2.6737e-03,  5.1121e-04,  ..., -3.4909e-01,
           4.4393e-02,  3.3815e-02],
         ...,
         [ 9.8995e-02,  1

In [13]:
src = outputs['last_hidden_state']
input_dim = src.size(-1)
seq_len = src.size(1)

In [114]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_size, dropout):
        super().__init__()
        # Projects hidden states to compute attention scores
        self.W = nn.LazyLinear(hidden_size, bias=False)
        self.v = nn.LazyLinear(1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden_states, attention_mask):
        """
        hidden_states: [batch_size, seq_len, hidden_size]
        attention_mask: [batch_size, seq_len]
        """
        # 1. Compute unnormalized attention scores
        # Shape: [batch_size, seq_len, 1] -> [batch_size, seq_len]
        scores = self.v(torch.tanh(self.W(hidden_states))).squeeze(-1)
        
        # 2. Mask out padding tokens so they don't receive attention
        # We use a large negative number (-1e9) so softmax outputs 0
        scores = scores.masked_fill(attention_mask == 0, -1e9)
        
        # 3. Normalize scores into probabilities (alpha weights)
        attn_weights = torch.softmax(scores, dim=-1) # [batch_size, seq_len]
        
        # 4. Compute the weighted sum of hidden states (Context Vector)
        # unsqueeze attn_weights to [batch, 1, seq_len] for batch matrix multiplication
        # output shape: [batch, 1, hidden_size] -> [batch, 1, hidden_size]
        context_vector = torch.bmm(self.dropout(attn_weights).unsqueeze(1), hidden_states)
        
        return context_vector, attn_weights

In [59]:
import math
def masked_softmax(X, valid_lens):  #@save
    """Perform softmax operation by masking elements on the last axis."""
    # X: 3D tensor, valid_lens: 1D or 2D tensor
    def _sequence_mask(X, valid_len, value=0):
        maxlen = X.size(1)
        mask = torch.arange((maxlen), dtype=torch.float32,
                            device=X.device)[None, :] < valid_len[:, None]
        X[~mask] = value
        return X

    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        # On the last axis, replace masked elements with a very large negative
        # value, whose exponentiation outputs 0
        X = _sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)

class DotProductAttention(nn.Module):  #@save
    """Scaled dot product attention."""
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    # Shape of queries: (batch_size, no. of queries, d)
    # Shape of keys: (batch_size, no. of key-value pairs, d)
    # Shape of values: (batch_size, no. of key-value pairs, value dimension)
    # Shape of valid_lens: (batch_size,) or (batch_size, no. of queries)
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        # Swap the last two dimensions of keys with keys.transpose(1, 2)
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values)


class MultiHeadAttention(nn.Module):  #@save
    """Multi-head attention."""
    def __init__(self, num_hiddens, num_heads, dropout, bias=False, **kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_k = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_v = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_o = nn.LazyLinear(num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # Shape of queries, keys, or values:
        # (batch_size, no. of queries or key-value pairs, num_hiddens)
        # Shape of valid_lens: (batch_size,) or (batch_size, no. of queries)
        # After transposing, shape of output queries, keys, or values:
        # (batch_size * num_heads, no. of queries or key-value pairs,
        # num_hiddens / num_heads)
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))

        if valid_lens is not None:
            # On axis 0, copy the first item (scalar or vector) for num_heads
            # times, then copy the next item, and so on
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # Shape of output: (batch_size * num_heads, no. of queries,
        # num_hiddens / num_heads)
        output = self.attention(queries, keys, values, valid_lens)
        # Shape of output_concat: (batch_size, no. of queries, num_hiddens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

    def transpose_qkv(self, X):
        """Transposition for parallel computation of multiple attention heads."""
        # Shape of input X: (batch_size, no. of queries or key-value pairs,
        # num_hiddens). Shape of output X: (batch_size, no. of queries or
        # key-value pairs, num_heads, num_hiddens / num_heads)
        X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
        # Shape of output X: (batch_size, num_heads, no. of queries or key-value
        # pairs, num_hiddens / num_heads)
        X = X.permute(0, 2, 1, 3)
        # Shape of output: (batch_size * num_heads, no. of queries or key-value
        # pairs, num_hiddens / num_heads)
        return X.reshape(-1, X.shape[2], X.shape[3])
    
    def transpose_output(self, X):
        """Reverse the operation of transpose_qkv."""
        X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
        X = X.permute(0, 2, 1, 3)
        return X.reshape(X.shape[0], X.shape[1], -1)

In [15]:
class PositionalEncoding(nn.Module): 
    """Positional encoding."""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # Create a long enough P
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)
        
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_seq_len=128, latent_dim=768):
        super().__init__()
        self.positional_embedding = nn.Embedding(max_seq_len, latent_dim)
    
    def forward(self, x):
        positions = torch.arange(x.size(-2)).expand(src.size(0), -1)
        embed = self.positional_embedding(positions)
        return x + embed

In [115]:
Learned=False
if not Learned:
    positional_encoder = PositionalEncoding(num_hiddens=input_dim, dropout=0.1, max_len=128)
    src_pos_embed = positional_encoder(src)
else:
    positional_encoder = LearnedPositionalEncoding(max_seq_len=128, latent_dim=input_dim)
    src_pos_embed = positional_encoder(src)

addAttn = AdditiveAttention(
    hidden_size = input_dim,
    dropout = 0.0
    )
selfAttn = nn.modules.activation.MultiheadAttention(
    embed_dim = input_dim, 
    num_heads = 1,
    batch_first = True)
selfAttn_d2l = MultiHeadAttention(
    num_hiddens = input_dim,
    num_heads = 1,
    dropout = 0.0,
    bias = True
)

In [17]:
Additive=True
if Additive:
    src_mask = data['attention_mask']
    agg_outputs, attn_weights = addAttn(src_pos_embed, src_mask)
else:
    src_mask = data['attention_mask'].to(torch.float).unsqueeze(-1).expand((-1,-1,seq_len))
    agg_outputs, attn_weights = selfAttn(src_pos_embed, src_pos_embed, src_pos_embed, attn_mask = src_mask)

In [116]:
src_mask = data['attention_mask']
agg_outputs, attn_weights = addAttn(src, src_mask)

In [117]:
agg_outputs[:, 0, :]

tensor([[-0.0377, -0.0352, -0.0129,  ..., -0.0286,  0.0516, -0.0059],
        [-0.0381, -0.0111,  0.0224,  ...,  0.0195,  0.0453,  0.0204],
        [-0.0362, -0.0278,  0.0031,  ...,  0.0316,  0.0535,  0.0209],
        ...,
        [-0.0066,  0.0004, -0.0064,  ...,  0.0048,  0.0325, -0.0090],
        [-0.0349,  0.0121,  0.0220,  ...,  0.0223,  0.0344,  0.0022],
        [-0.0292,  0.0007, -0.0055,  ..., -0.0530,  0.0489,  0.0563]],
       grad_fn=<SliceBackward0>)

In [118]:
agg_outputs.shape

torch.Size([64, 1, 768])

In [36]:
src_mask[0,:]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [75]:
attn_weights[0,:]

tensor([0.0416, 0.0406, 0.0415, 0.0409, 0.0399, 0.0383, 0.0410, 0.0417, 0.0400,
        0.0396, 0.0386, 0.0373, 0.0397, 0.0400, 0.0407, 0.0418, 0.0378, 0.0398,
        0.0408, 0.0397, 0.0395, 0.0396, 0.0398, 0.0389, 0.0408, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SliceBackward0>)

In [85]:
src_mask = ~data['attention_mask'].to(torch.bool)
agg_outputs, attn_weights = selfAttn(src, src, src, key_padding_mask  = src_mask)

In [87]:
agg_outputs

tensor([[[-0.4490,  0.1822, -0.0459,  ...,  0.0383,  0.6791,  0.0314],
         [-0.4488,  0.1823, -0.0462,  ...,  0.0381,  0.6789,  0.0313],
         [-0.4487,  0.1824, -0.0466,  ...,  0.0384,  0.6791,  0.0312],
         ...,
         [-0.4491,  0.1822, -0.0460,  ...,  0.0384,  0.6793,  0.0314],
         [-0.4491,  0.1822, -0.0460,  ...,  0.0384,  0.6793,  0.0314],
         [-0.4491,  0.1822, -0.0460,  ...,  0.0384,  0.6793,  0.0314]],

        [[-0.4335,  0.1620, -0.0188,  ...,  0.0147,  0.6842,  0.0604],
         [-0.4330,  0.1620, -0.0192,  ...,  0.0141,  0.6837,  0.0603],
         [-0.4333,  0.1624, -0.0197,  ...,  0.0149,  0.6844,  0.0604],
         ...,
         [-0.4336,  0.1621, -0.0189,  ...,  0.0148,  0.6843,  0.0605],
         [-0.4336,  0.1621, -0.0189,  ...,  0.0148,  0.6843,  0.0605],
         [-0.4336,  0.1621, -0.0189,  ...,  0.0148,  0.6843,  0.0605]],

        [[-0.4563,  0.1848, -0.0414,  ...,  0.0392,  0.6807,  0.0223],
         [-0.4559,  0.1848, -0.0419,  ...,  0

In [44]:
src_mask[0,0,:]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [86]:
attn_weights[0, 0, :]

tensor([0.0309, 0.0389, 0.0427, 0.0403, 0.0418, 0.0410, 0.0391, 0.0422, 0.0414,
        0.0409, 0.0408, 0.0417, 0.0414, 0.0406, 0.0386, 0.0416, 0.0411, 0.0412,
        0.0409, 0.0403, 0.0396, 0.0407, 0.0405, 0.0416, 0.0303, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       grad_fn=<SliceBackward0>)

In [88]:
valid_lens = data['attention_mask'].sum(1)
agg_outputs = selfAttn_d2l(src, src, src, valid_lens = valid_lens)

In [91]:
selfAttn_d2l.attention.attention_weights[0, :, :]

tensor([[0.0376, 0.0400, 0.0419,  ..., 0.0000, 0.0000, 0.0000],
        [0.0364, 0.0411, 0.0419,  ..., 0.0000, 0.0000, 0.0000],
        [0.0354, 0.0411, 0.0420,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0382, 0.0397, 0.0417,  ..., 0.0000, 0.0000, 0.0000],
        [0.0382, 0.0397, 0.0417,  ..., 0.0000, 0.0000, 0.0000],
        [0.0382, 0.0397, 0.0417,  ..., 0.0000, 0.0000, 0.0000]],
       grad_fn=<SliceBackward0>)

In [97]:
agg_outputs[:, 0, :].shape

torch.Size([64, 768])